# EEdit — Ablation Study

Runs **4 ablation conditions** on **10 images per domain** (40 images/condition, 160 total).

| Condition | KV Injection | Prompt Aug | Tail CFG | What changes |
|-----------|:---:|:---:|:---:|---|
| `no_kv` | ❌ | ✅ | ✅ | `ref_inject_blocks/steps` removed from config |
| `no_prompt_aug` | ✅ | ❌ | ✅ | `--no_prompt_aug` flag; full configs used |
| `no_tail_cfg` | ✅ | ✅ | ❌ | `cfg_tail_steps/cfg_scale` removed from config |
| `no_all` | ❌ | ❌ | ❌ | all three removed; pure EEdit + our eta/gamma |

eta/gamma = 0.75 across **all** conditions so it is not a confounding variable.

**Run order:** Clone → Pip install → Restart runtime → all remaining cells in order.

In [1]:
import subprocess, os, shutil

!nvidia-smi || true

GITHUB_USER = "AimeeAyat"
REPO_NAME   = "FIA-EDIT-DVLM"
BRANCH      = "EEdit_baseline"
repo_url    = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

def run_cmd(cmd):
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"\nERROR: {' '.join(cmd)}\n{e.stderr}")
        raise

if os.path.exists("/content/EEdit"):
    if not os.path.exists("/content/EEdit/.git"):
        shutil.rmtree("/content/EEdit")
        run_cmd(["git", "clone", repo_url, "/content/EEdit"])
    else:
        run_cmd(["git", "-C", "/content/EEdit", "pull"])
else:
    run_cmd(["git", "clone", repo_url, "/content/EEdit"])

%cd /content/EEdit
!git checkout {BRANCH}
!python -c "import dvlm; print('dvlm OK')" || echo 'ERROR: dvlm/ not found'

Sat May 30 11:50:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             49W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%pip uninstall -y numpy opencv-python opencv-python-headless || true
%pip install -q --upgrade pip
%pip install -q --upgrade --force-reinstall --no-cache-dir \
  numpy==1.26.4 \
  torch==2.5.1 torchvision==0.20.1 xformers==0.0.28.post3 \
  diffusers==0.31.0 transformers==4.46.1 accelerate==1.1.0 \
  sentencepiece==0.2.0 safetensors==0.5.2 huggingface_hub==0.26.2 \
  ftfy==6.3.1 einops==0.8.1 omegaconf==2.3.0 \
  pillow==11.1.0 opencv-python-headless==4.10.0.84 \
  gdown lpips torchmetrics
print("Install complete. IMPORTANT: Runtime -> Restart Session")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have r

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
%cd /content/EEdit
import numpy, torch, transformers
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
!python -c "import dvlm; print('dvlm OK')"

/content/EEdit
numpy: 1.26.4
torch: 2.5.1+cu124
transformers: 4.46.1
CUDA: True NVIDIA A100-SXM4-80GB
dvlm OK


In [2]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from datetime import datetime
from pathlib import Path
import os

auth.authenticate_user()
drive_service = build("drive", "v3")

INPUT_FILE_ID    = "1U7BIJZZinAzraAt_T8jAKX3uPa5c-HQa"  # composition subset ZIP
OUTPUT_FOLDER_ID = "1R0DNBTzobpIOyYDZuy5YZ2lc599nTu8Q"  # Drive output folder

_ts = datetime.now().strftime("%Y%m%d_%H%M")
RUN_FOLDER_ID = None

def _get_run_folder():
    global RUN_FOLDER_ID
    if RUN_FOLDER_ID is None:
        RUN_FOLDER_ID = _gdrive_mkdir(drive_service, f"EEdit_ablation_{_ts}", OUTPUT_FOLDER_ID)
        print(f"Created run folder: EEdit_ablation_{_ts}")
    return RUN_FOLDER_ID

def _gdrive_mkdir(service, name, parent_id):
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return service.files().create(body=meta, fields="id").execute()["id"]

def _gdrive_upload_file(service, local_path, parent_id):
    meta = {"name": os.path.basename(local_path), "parents": [parent_id]}
    media = MediaFileUpload(local_path, resumable=True)
    service.files().create(body=meta, media_body=media, fields="id").execute()

def _upload_tree(service, local_dir, parent_id):
    total = 0
    for item in sorted(Path(local_dir).iterdir()):
        if item.is_dir():
            child_id = _gdrive_mkdir(service, item.name, parent_id)
            total += _upload_tree(service, str(item), child_id)
        else:
            _gdrive_upload_file(service, str(item), parent_id)
            total += 1
    return total

def upload_ablation_results(ablation_name, generated_dir):
    abl_id = _gdrive_mkdir(drive_service, ablation_name, _get_run_folder())
    n = _upload_tree(drive_service, generated_dir, abl_id)
    print(f"  [{ablation_name}] uploaded {n} files")

print("Google Drive authenticated.")

Google Drive authenticated.


In [3]:
import os, shutil, zipfile
from pathlib import Path

COMPOSITION_DIR = "/content/TF-ICON/inputs"
os.makedirs(COMPOSITION_DIR, exist_ok=True)

def gdrive_download(service, file_id, dest_path):
    req = service.files().get_media(fileId=file_id)
    with open(dest_path, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64*1024*1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            print(f"  {status.progress()*100:.0f}%", end="\r")
    print(f"  Downloaded: {os.path.basename(dest_path)}")

def extract_zip(zip_path, dest_dir):
    tmp = zip_path + "_tmp"
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)
    items = [x for x in os.listdir(tmp) if x != "__MACOSX"]
    src = (os.path.join(tmp, items[0])
           if len(items) == 1 and os.path.isdir(os.path.join(tmp, items[0]))
           else tmp)
    for item in os.listdir(src):
        dst = os.path.join(dest_dir, item)
        if os.path.exists(dst):
            shutil.rmtree(dst) if os.path.isdir(dst) else os.remove(dst)
        shutil.move(os.path.join(src, item), dst)
    shutil.rmtree(tmp, ignore_errors=True)

meta = drive_service.files().get(fileId=INPUT_FILE_ID, fields="name,size").execute()
fname = meta["name"]
mb    = int(meta.get("size", 0)) / 1024**2
print(f"Downloading {fname}  ({mb:.1f} MB) ...")
local_zip = f"/content/{fname}"
gdrive_download(drive_service, INPUT_FILE_ID, local_zip)
print(f"Extracting to {COMPOSITION_DIR} ...")
extract_zip(local_zip, COMPOSITION_DIR)
os.remove(local_zip)

n = sum(1 for _ in Path(COMPOSITION_DIR).rglob("*") if _.is_file())
print(f"\nComposition input: {n} files")
for p in sorted(Path(COMPOSITION_DIR).iterdir()):
    if p.is_dir():
        count = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  {p.name}/  ({count} files)")

  Downloaded: composition-test.zip
Extracting to /content/TF-ICON/inputs ...

Composition input: 53 files
  Real-Cartoon/  (13 files)
  Real-Painting/  (13 files)
  Real-Real/  (13 files)
  Real-Sketch/  (13 files)


## 1. Hugging Face Weights

Accept the FLUX.1-dev access agreement first: https://huggingface.co/black-forest-labs/FLUX.1-dev

Then run this cell with your HF token.

In [4]:
from huggingface_hub import login, snapshot_download
from getpass import getpass
import os

token = getpass("Hugging Face token: ")
login(token=token)

os.makedirs("/content/EEdit/weights", exist_ok=True)

snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    local_dir="/content/EEdit/weights",
    allow_patterns=[
        "flux1-dev.safetensors",
        "transformer/config.json",
        "transformer_config.json",
        "model_index.json",
        "scheduler/*",
        "text_encoder/*",
        "text_encoder_2/*",
        "tokenizer/*",
        "tokenizer_2/*",
        "vae/*",
    ],
)
print("Weights downloaded.")

Hugging Face token: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

flux1-dev.safetensors:   0%|          | 0.00/23.8G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/820 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Weights downloaded.


## 2. Data Setup — 10 images per domain

Same 10 images are used for **every** ablation condition so results are directly comparable.

In [5]:
import os, re, json, shutil, numpy as np
from pathlib import Path
from PIL import Image

SUBSET_PER_GROUP = 10   # same 10 images for every ablation condition

tf_root    = Path("/content/TF-ICON/inputs")
eedit_root = Path("/content/EEdit/input/composition")
cfg_root   = Path("/content/EEdit/configs/ablation")   # separate from subset/full configs
eedit_root.mkdir(parents=True, exist_ok=True)
cfg_root.mkdir(parents=True, exist_ok=True)

groups = {"Real-Cartoon": [], "Real-Painting": [], "Real-Sketch": [], "Real-Real": []}

def pick(files, patterns, exclude=()):
    for pat in patterns:
        for f in files:
            if re.match(pat, f.name.lower()) and not any(e in f.name.lower() for e in exclude):
                return f
    return None

skipped = 0
for group_name in ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]:
    domain_path = tf_root / group_name
    if not domain_path.exists(): continue
    collected = 0
    for idx, sample_dir in enumerate(sorted(domain_path.iterdir())):
        if collected >= SUBSET_PER_GROUP: break
        if not sample_dir.is_dir(): continue
        prompt = sample_dir.name
        files  = [p for p in sample_dir.iterdir() if p.is_file()]
        bg         = pick(files, [r"^bg.*\.(jpg|jpeg|png)$"])
        ref_img    = pick(files, [r"^fg.*\.(jpg|jpeg|png)$", r"^dccf.*\.jpg$"], exclude=("_mask",))
        ref_mask   = pick(files, [r"^fg.*_mask.*\.(png|jpg)$"])
        place_mask = pick(files, [r"^mask_bg_fg.*\.(jpg|png)$"])
        if not all([bg, ref_img, place_mask]):
            skipped += 1; continue

        slug    = f"{group_name[:2]}_{idx:04d}_{prompt[:60]}"
        out_dir = eedit_root / group_name / slug
        out_dir.mkdir(parents=True, exist_ok=True)
        for src in [bg, ref_img, place_mask] + ([ref_mask] if ref_mask else []):
            if src: shutil.copy2(src, out_dir / src.name)

        arr = np.array(Image.open(place_mask).convert("L"))
        ys, xs = np.where(arr > 10)
        if len(xs) == 0: skipped += 1; continue
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())

        groups[group_name].append({
            "prompt":      prompt,
            "main_image":  str(out_dir / bg.name).replace("/content/EEdit/", "./"),
            "ref_image":   str(out_dir / ref_img.name).replace("/content/EEdit/", "./"),
            "ref_segment": str(out_dir / (ref_mask.name if ref_mask else ref_img.name)).replace("/content/EEdit/", "./"),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2
        })
        collected += 1

for group, imgs in groups.items():
    out_json = cfg_root / f"{group}.json"
    with open(out_json, "w") as f:
        json.dump({"imgs": imgs}, f, indent=2)
    print(f"{group}: {len(imgs)} samples -> {out_json}")
if skipped:
    print(f"({skipped} samples skipped — missing required files)")

Real-Cartoon: 2 samples -> /content/EEdit/configs/ablation/Real-Cartoon.json
Real-Painting: 2 samples -> /content/EEdit/configs/ablation/Real-Painting.json
Real-Sketch: 2 samples -> /content/EEdit/configs/ablation/Real-Sketch.json
Real-Real: 2 samples -> /content/EEdit/configs/ablation/Real-Real.json


## 3. Run All Ablation Conditions

Each condition runs on the same 10 images per domain.
Results are uploaded to Drive after each condition completes.

**Ablation conditions and what they isolate:**

- `no_kv`: removes reference K,V injection → shows its contribution to identity preservation
- `no_prompt_aug`: removes domain-style prompt suffixes → shows their effect on style integration
- `no_tail_cfg`: removes tail classifier-free guidance → shows its effect on style sharpening
- `no_all`: removes all three → closest to base EEdit (only eta/gamma tuning remains)

In [6]:
%cd /content/EEdit
import gc, torch, json, os, subprocess
from datetime import datetime
from pathlib import Path
gc.collect(); torch.cuda.empty_cache()

RUN_TAG     = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = f"EEdit_outputs/ablation_{RUN_TAG}"
print(f"Output root: {OUTPUT_ROOT}")

# ── Ablation conditions ───────────────────────────────────────────────────────
# Each tuple: (condition_name, config_dir, extra_cli_flags)
#
# no_kv:         ablation config without ref_inject_blocks/steps
#                tail CFG still active (--use_tail_cfg)
#
# no_prompt_aug: full domain configs, but --no_prompt_aug disables augmented prompts
#                tail CFG still active (--use_tail_cfg)
#
# no_tail_cfg:   ablation config without cfg_tail_steps/cfg_scale
#                --use_tail_cfg NOT passed (no-op, but omitted for clarity)
#
# no_all:        ablation config with neither ref_inject nor cfg_tail
#                --no_prompt_aug also passed to disable all three additions
ABLATION_CONDITIONS = [
    ("no_kv",         "dvlm/ablation_configs/no_kv",       "--use_tail_cfg"),
    ("no_prompt_aug", "dvlm/domain_configs",                "--use_tail_cfg --no_prompt_aug"),
    ("no_tail_cfg",   "dvlm/ablation_configs/no_tail_cfg",  ""),
    ("no_all",        "dvlm/ablation_configs/no_all",       "--no_prompt_aug"),
]

DOMAIN_CFG_MAP = {
    "Real-Cartoon":  "RC_config.json",
    "Real-Painting": "RP_config.json",
    "Real-Sketch":   "RS_config.json",
    "Real-Real":     "RR_config.json",
}

for ablation_name, config_dir, extra_flags in ABLATION_CONDITIONS:
    print(f"\n{'='*60}")
    print(f"  ABLATION: {ablation_name}")
    print(f"{'='*60}")
    abl_out_root = f"{OUTPUT_ROOT}/{ablation_name}"

    for group, cfg_file in DOMAIN_CFG_MAP.items():
        img_cfg = f"/content/EEdit/configs/ablation/{group}.json"
        if not os.path.exists(img_cfg):
            print(f"  Skipping {group} — no img config"); continue
        with open(img_cfg) as f:
            n = len(json.load(f)["imgs"])
        if n == 0:
            print(f"  Skipping {group} — 0 images"); continue

        out_dir = f"{abl_out_root}/{group}"
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n  --- {group} ({n} images) -> {out_dir} ---")

        cmd = (
            f"python dvlm/composition_gen.py"
            f" --weights_dir ./weights"
            f" --config_path ./{config_dir}/{cfg_file}"
            f" --img_config  ./configs/ablation/{group}.json"
            f" --output_dir  ./{out_dir}"
            f" --use_predefine 1"
            f" {extra_flags}"
        )
        os.system(cmd)

    # Upload this condition's results to Drive before moving on
    print(f"\n  Uploading {ablation_name} results to Drive...")
    upload_ablation_results(f"{ablation_name}_{RUN_TAG}", abl_out_root)

print("\n\nAll ablation conditions complete.")
total = sum(
    len(list(Path(f"{OUTPUT_ROOT}/{abl}/{grp}").glob("*.png")))
    for abl, _, _ in ABLATION_CONDITIONS
    for grp in DOMAIN_CFG_MAP
    if Path(f"{OUTPUT_ROOT}/{abl}/{grp}").exists()
)
print(f"Total images generated: {total}  (expected: {len(ABLATION_CONDITIONS) * len(DOMAIN_CFG_MAP) * SUBSET_PER_GROUP})")

/content/EEdit
Output root: EEdit_outputs/ablation_20260530_115555

  ABLATION: no_kv

  --- Real-Cartoon (2 images) -> EEdit_outputs/ablation_20260530_115555/no_kv/Real-Cartoon ---

  --- Real-Painting (2 images) -> EEdit_outputs/ablation_20260530_115555/no_kv/Real-Painting ---

  --- Real-Sketch (2 images) -> EEdit_outputs/ablation_20260530_115555/no_kv/Real-Sketch ---

  --- Real-Real (2 images) -> EEdit_outputs/ablation_20260530_115555/no_kv/Real-Real ---

  Uploading no_kv results to Drive...
Created run folder: EEdit_ablation_20260530_1153
  [no_kv_20260530_115555] uploaded 12 files

  ABLATION: no_prompt_aug

  --- Real-Cartoon (2 images) -> EEdit_outputs/ablation_20260530_115555/no_prompt_aug/Real-Cartoon ---

  --- Real-Painting (2 images) -> EEdit_outputs/ablation_20260530_115555/no_prompt_aug/Real-Painting ---

  --- Real-Sketch (2 images) -> EEdit_outputs/ablation_20260530_115555/no_prompt_aug/Real-Sketch ---

  --- Real-Real (2 images) -> EEdit_outputs/ablation_20260530_11

## 4. Image Count Summary

In [7]:
from pathlib import Path

ABLATION_NAMES = ["no_kv", "no_prompt_aug", "no_tail_cfg", "no_all"]
GROUPS         = ["Real-Cartoon", "Real-Painting", "Real-Sketch", "Real-Real"]

print(f"{'Condition':<18} ", end="")
for g in GROUPS:
    print(f"{g[:12]:>13}", end="")
print(f"{'Total':>8}")
print("-" * 75)

for abl in ABLATION_NAMES:
    counts = []
    for grp in GROUPS:
        d = Path(f"{OUTPUT_ROOT}/{abl}/{grp}")
        counts.append(len(list(d.glob("*.png"))) if d.exists() else 0)
    print(f"{abl:<18} ", end="")
    for c in counts:
        print(f"{c:>13}", end="")
    print(f"{sum(counts):>8}")

print("-" * 75)
print(f"Output root: {OUTPUT_ROOT}")

Condition           Real-Cartoon Real-Paintin  Real-Sketch    Real-Real   Total
---------------------------------------------------------------------------
no_kv                          2            2            2            2       8
no_prompt_aug                  2            2            2            2       8
no_tail_cfg                    2            2            2            2       8
no_all                         2            2            2            2       8
---------------------------------------------------------------------------
Output root: EEdit_outputs/ablation_20260530_115555
